# Task 1.2 — Data Quality Hunt (15 min)
`bank_marketing_dirty.csv` hides **five** injected defects. Find **at least 3**.
For each: **(1) evidence** (the output that reveals it), **(2) a concrete fix**, **(3) why it matters for training**.
Useful moves: `.info()`, `.describe()`, `.value_counts()`, `.duplicated().sum()`.

In [2]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [3]:
from course_utils import data
df = data.load_raw('bank_marketing_dirty')
df.head()

,age,job,marital,education,balance,housing,loan,contact,month,duration,campaign,y
0,85,admin.,divorced,tertiary,1763,yes,no,unknown,dec,1850,11,no
1,83,student,divorced,unknown,1913,no,yes,cellular,jul,806,13,no
2,70,retired,married,primary,-716,yes,yes,cellular,aug,1404,14,no
3,46,blue-collar,married,secondary,2239,yes,no,telephone,Mar,2211,3,yes
4,60,retired,single,primary,-578,no,no,telephone,may,516,2,no


In [4]:
# your exploration here
print(df.shape)
df.info()

(3200, 12)
<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        3200 non-null   int64
 1   job        3200 non-null   str  
 2   marital    3200 non-null   str  
 3   education  3200 non-null   str  
 4   balance    3200 non-null   int64
 5   housing    3200 non-null   str  
 6   loan       3200 non-null   str  
 7   contact    3200 non-null   str  
 8   month      3200 non-null   str  
 9   duration   3200 non-null   int64
 10  campaign   3200 non-null   int64
 11  y          3200 non-null   str  
dtypes: int64(4), str(8)
memory usage: 300.1 KB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,3200.0,53.493750,35.211059,18.0,35.00,52.0,70.0,999.0
balance,3200.0,1375.736875,2756.358584,-8093.0,-491.50,1389.5,3157.5,11687.0
duration,3200.0,1480.239063,868.438330,1.0,731.75,1481.5,2242.5,2999.0
campaign,3200.0,7.445000,4.040136,1.0,4.00,7.0,11.0,14.0


In [6]:
(df['age'] == 999).sum()
df['age'].sort_values().tail(10)

1687     87
1656     87
1297     87
1802     87
109      87
2794     87
2370     87
2189    999
1999    999
3021    999
Name: age, dtype: int64

In [ ]:
df = df[df['age'] != 999] #drops the rows with age=999

In [9]:
df = data.load_raw('bank_marketing_dirty')
import numpy as np
df['age'] = df['age'].replace(999, np.nan)
df['age'] = df['age'].fillna(df['age'].median())
print(df['age'].max(), len(df))   # should print: 87 3200

87.0 3200


In [ ]:
df.duplicated().sum() #check how many duplicates

np.int64(136)

In [11]:
df = df.drop_duplicates()
print(df.duplicated().sum(), len(df))

0 3064


In [12]:
df['month'].value_counts()

month
mar    220
may    219
sep    219
feb    218
jun    217
oct    217
nov    204
aug    198
jan    198
dec    192
apr    191
jul    176
Mar     54
Oct     44
Jun     42
Nov     39
Aug     38
Jan     37
Dec     36
Feb     34
Sep     33
Apr     29
Jul     26
May     24
JUL     19
APR     17
AUG     17
MAY     16
FEB     14
SEP     14
NOV     14
JAN     13
JUN     11
OCT      9
DEC      8
MAR      7
Name: count, dtype: int64

In [14]:
valid = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
bad = df.loc[~df['month'].isin(valid), 'month']
print(len(bad))
bad.value_counts()
print(bad.unique())

595
<StringArray>
['Mar', 'JUL', 'APR', 'Sep', 'DEC', 'Oct', 'Apr', 'Feb', 'Nov', 'FEB', 'Jan',
 'Jun', 'May', 'Jul', 'MAR', 'SEP', 'JAN', 'Dec', 'NOV', 'Aug', 'JUN', 'AUG',
 'MAY', 'OCT']
Length: 24, dtype: str


In [15]:
df['month'] = df['month'].str.strip().str.lower() #fix
df['month'].nunique()  

12

## Findings
| # | Issue | Evidence | Fix | Why it matters |
|---|---|---|---|---|
| 1 | Sentinel value 999 in age | describe() → age max = 999; (df['age']==999).sum() | replace with the median |  The model treats 999 as a real, very old age. That distorts the mean and the scaling and teaches the model a false relationship between age and y|
| 2 | Duplicate rows |  df.duplicated().sum() → 136|  df = df.drop_duplicates()|  Duplicates overweight some examples and after the train/test split the same row can land in both sets, so the model is tested on data it already saw and the score is inflated|
| 3 | Inconsistent capitalization in month | value_counts() → 36 unique values instead of 12; 595 rows outside the valid list (Mar, Oct, ...) | value_counts() → 36 unique values instead of 12; 595 rows outside the valid list (Mar, Oct, ...) | After one-hot encoding, mar and Mar become separate features, so information about the same month is split and the model learns a weaker pattern |